## # BRONZE EXPLORATION START

In [0]:
test_df = fetch_ticker_history_range("AAPL", "2024-01-01", "2024-06-01")
test_df.head()

In [0]:
raw_pdf["ticker"].value_counts()
raw_pdf[raw_pdf["ticker"] == "COST"]["date"].agg(["min", "max", "count"])

raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"].agg(["min", "max", "count"])
aapl_dates = set(raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"])
cost_dates = set(raw_pdf[raw_pdf["ticker"] == "COST"]["date"])
aapl_dates - cost_dates

In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SHOW TABLES IN mashup_learning.stocks


In [0]:
%sql
DESCRIBE TABLE mashup_learning.stocks.bronze_daily_prices;


In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.bronze_daily_prices;
    


In [0]:
%sql DESCRIBE HISTORY mashup_learning.stocks.bronze_daily_prices

In [0]:
%sql
SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, 0) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close = 0 then 0 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices;
--WHERE TICKER = 'NVDA'
--ORDER BY DATE ASC;

%md
## # SILVER EXPLORATION START

In [0]:
%sql
SELECT * FROM (SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, NULL) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close is null then null 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
case WHEN
COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)<7 THEN NULL 
ELSE AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  end AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices)
WHERE ticker = 'AAPL'
ORDER BY "date"
LIMIT 10;

In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.silver_daily_prices;

In [0]:
%sql
SELECT *,
AVG(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS rolling_avg_volume_20d,
case WHEN
COUNT(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)<20 THEN NULL 
else volume/AVG(volume) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) end AS relative_volume_20d
 FROM 
mashup_learning.stocks.bronze_daily_prices
WHERE ticker = 'AAPL'
ORDER BY "date"



## # GOLD EXPLORATION START

In [0]:
%sql
SELECT trade_date,previous_day_close , daily_return,rolling_avg_7d,open , high, low, close, volume, dividends, stock_splits, ticker, CURRENT_TIMESTAMP() AS gold_processed_at ,
CASE 
  WHEN COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL 
  ELSE
AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)  end AS rolling_avg_volume_20d,
CASE 
  WHEN COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL 
  ELSE volume / rolling_avg_volume_20d 
END AS relative_volume_20d,
CASE 
WHEN COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL
else STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW ) end as return_volatility_20d,
case WHEN COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) < 20 THEN NULL
else AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) end as swing_volatility_20d
FROM mashup_learning.stocks.silver_daily_prices
WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25

In [0]:
%sql
WITH staged AS (
  SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_count_20d,
    AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS raw_avg_volume_20d,
    COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) as ret_count_20d,
    COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS row_count_20d
    -- other raw window calcs here
  FROM mashup_learning.stocks.silver_daily_prices
  WHERE ticker = 'AAPL'
ORDER BY trade_date
LIMIT 25
)
SELECT
  trade_date, previous_day_close, daily_return, rolling_avg_7d,
  open, high, low, close, volume, dividends, stock_splits, ticker,
  CURRENT_TIMESTAMP() AS gold_processed_at,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE raw_avg_volume_20d END AS rolling_avg_volume_20d,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE volume / raw_avg_volume_20d END AS relative_volume_20d,
  CASE WHEN ret_count_20d < 20 THEN NULL ELSE STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW )end as return_volatility_20d,
  CASE WHEN row_count_20d < 20 THEN NULL ELSE AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) END as swing_volatility_20d
FROM staged;

In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.gold_daily_metrics;


In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.gold_daily_metrics;

In [0]:
%sql
SELECT ticker, MIN(trade_date), MAX(trade_date),
  SUM(CASE WHEN swing_volatility_20d IS NULL THEN 1 ELSE 0 END) AS null_swing_count,
  SUM(CASE WHEN return_volatility_20d IS NULL THEN 1 ELSE 0 END) AS null_return_vol_count
FROM mashup_learning.stocks.gold_daily_metrics
GROUP BY ticker
ORDER BY ticker;